# BAM PatchTST Walk-Forward Training

Trains PatchTST alpha model on Google Colab T4 GPU.

**Walk-forward folds:**
- Fold 1: train 2010-2014, val 2015, test 2016
- Fold 9: train 2018-2022, val 2023, test 2024

**Estimated time:** ~30 min/fold on T4 GPU, ~4.5 hours total for all 9 folds.

In [ ]:
# Step 1: Mount Google Drive and clone repo
from google.colab import drive
drive.mount('/content/drive')

# Clone or pull the BAM repo
import os
REPO_DIR = '/content/BAM'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    # Upload the repo as a zip or clone from GitHub
    # Option A: Upload zip from Drive
    !cp /content/drive/MyDrive/BAM.zip /content/ && unzip -q /content/BAM.zip -d /content/
    # Option B: Clone from GitHub (if pushed)
    # !git clone https://github.com/YOUR_USER/BAM.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Step 2: Install dependencies
!pip install -q torch numpy pandas scipy wandb pyarrow

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Step 3: Verify data files exist
from pathlib import Path

PROCESSED_DIR = Path('alpha_model/data/processed')
required_files = ['features.parquet', 'targets.parquet', 'ticker_meta.parquet']

for f in required_files:
    path = PROCESSED_DIR / f
    if path.exists():
        import pandas as pd
        df = pd.read_parquet(path)
        print(f'{f}: {df.shape}')
    else:
        print(f'MISSING: {f} — run data pipeline first!')

In [ ]:
# Step 4: Optional — Login to W&B
# import wandb
# wandb.login(key='YOUR_WANDB_KEY')

In [ ]:
# Step 5: Train fold 9 (most recent, used for inference)
!python -m alpha_model.training.train_patchtst \
    --fold 9 \
    --device cuda \
    --epochs 50 \
    --batch-size 256

In [ ]:
# Step 6: (Optional) Train all 9 folds for complete walk-forward evaluation
# !python -m alpha_model.training.train_patchtst \
#     --all-folds \
#     --device cuda \
#     --epochs 50 \
#     --batch-size 256

In [ ]:
# Step 7: Evaluate the trained model
!python -m alpha_model.evaluation.evaluate \
    --model models/patch_tst_fold9.pt \
    --test-year 2024 \
    --device cuda

In [ ]:
# Step 8: Copy trained model to Google Drive for download to M1 Mac
import shutil

DRIVE_DIR = Path('/content/drive/MyDrive/BAM_models')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

model_path = Path('models/patch_tst_fold9.pt')
if model_path.exists():
    shutil.copy(model_path, DRIVE_DIR / 'patch_tst_fold9.pt')
    print(f'Copied to {DRIVE_DIR / "patch_tst_fold9.pt"}')
    print(f'Size: {model_path.stat().st_size / 1e6:.1f} MB')
else:
    print('Model not found — check training output')

In [ ]:
# Step 9: Quick inference test
from alpha_model.model.alpha_model import AlphaModel
import torch

model = AlphaModel.load('models/patch_tst_fold9.pt', device='cuda')
print(f'Parameters: {model.count_parameters():,}')

# Dummy input
ts = torch.randn(1, 23, 250).cuda()
sector = torch.zeros(1, 11).cuda()
sector[0, 7] = 1.0  # IT
cap = torch.zeros(1, 5).cuda()
cap[0, 4] = 1.0  # mega

result = model.predict(ts, sector, cap)
print(f'\nPrediction results:')
for k, v in result.items():
    print(f'  {k}: {v:.6f}' if isinstance(v, float) else f'  {k}: {v}')